# Data Cleaning/ Text Preprocessing of Montra Hospitality Intel

In [4]:
%pip install emoji
%pip install langdetect
%pip install great_expectations
%pip install nltk

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached regex-2026.9.29-cp312-cp312-win_amd64.whl.metadata (41 kB)
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------- ----- 1.6/1.8 MB 16.8 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 9.8 MB/s  0:00:00
Using cached regex-2026.9.29-cp312-cp312-win_amd64.whl (280 kB)
Using cached click-8.5.0-py3-none-any.whl (125 kB)

   ---------------------------------------- 0/3 [regex]
   ---------------------------------------- 0/3 [regex]
   ---------------------------------------- 0/3 [regex]
   ------------- -------------------------- 1/3 [click]
   ------------- -------------------------- 1/3 [click]
   ------------- -------------------------- 1/3 [cl

#### Import all Libraries

In [5]:
# Importing libraries
import os 
import pandas as pd
import plotly
from sqlalchemy import create_engine, text
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.express as px
import html
import re
import plotly.graph_objects as go
import plotly.subplots as sub
from plotly.subplots import make_subplots as msub 
import warnings; warnings.filterwarnings("ignore")
import json
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from turtle import title
import emoji
import nltk

from langdetect import detect, DetectorFactory
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
import great_expectations as ge
from sklearn.model_selection import GroupShuffleSplit

nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)
DetectorFactory.seed = 0
LABELS = ["negative", "neutral", "positive"]
COLORS = {"negative": "#D1495B", "neutral": "#B8BCC4", "positive": "#2E8B57"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

#### Load Cleaned Montra Dataset

In [6]:
# Load merged data file
merged_data = pd.read_csv(
    r"C:\Users\User\Documents\AMDARI_IMS\Montra_Hospitality_Intel_PJ\Dataset\raw_dataset\merged_montra_data.csv")
 
#  # converting the datetime
# date_cols = ["review_date", "check_in_date", "check_out_date"]

# for col in date_cols:
#     cleaned_data[col] = pd.to_datetime(cleaned_data[col], errors="coerce")

# Data Summary
data_summary = pd.DataFrame({
    "Data Type": merged_data.dtypes.astype(str),
    "Missing Values": merged_data.isna().sum(),
    "% Missing Values": (merged_data.isna().mean() * 100).round(2),
    "Unique Values": merged_data.nunique(),
    "Duplicate Values": [merged_data.duplicated(subset=[col]).sum() for col in merged_data.columns]
}).sort_values(by="% Missing Values", ascending=False)

data_summary

,Data Type,Missing Values,% Missing Values,Unique Values,Duplicate Values
loyalty_status,str,17615,50.33,3,34996
review_id,int64,0,0.00,35000,0
guest_id,int64,0,0.00,11377,23623
booking_id,int64,0,0.00,35000,0
review_text,str,0,0.00,34536,464
rating_score,float64,0,0.00,41,34959
review_date,str,0,0.00,545,34455
hotel_id,int64,0,0.00,25,34975
sentiment_label,str,0,0.00,3,34997
guest_id_booking_data,int64,0,0.00,11377,23623


### Structured Data cleaning

In [7]:
# ----------------------------------------------------------
# Check missing values 
# ----------------------------------------------------------
'''
Create a dataframe to show total number of missing values
and percentage of missing values 
'''

missing_data = pd.DataFrame({"Missing Data": merged_data.isnull().sum(), "Missing Percentage": (merged_data.isnull().mean() * 100).round(2)})

# Show only columns that contain missing values
missing_data = (missing_data[missing_data["Missing Data"] > 0].sort_values("Missing Percentage", ascending=False))

print("Missing values BEFORE treatment:")
print("--"*25)
print(missing_data)
print()


# Visualizing the Missing Data
print("Visaulization of Missing values BEFORE treatment:")
print("--"*25)
fig = px.bar(missing_data, x=missing_data.index, y="Missing Percentage",
             title="Percentage Missing Data",
             labels={"x": "Features", "Missing Percentage": "Missing Percentage (%)"},
             text="Missing Percentage")

fig.update_traces( texttemplate="%{text:.2f}%", textposition="outside")

fig.update_layout(showlegend=False, plot_bgcolor="white", paper_bgcolor="white",
                  xaxis=dict(showgrid=False, tickangle=-45), yaxis=dict(showgrid=False))

fig.show()

Missing values BEFORE treatment:
--------------------------------------------------
                Missing Data  Missing Percentage
loyalty_status         17615               50.33

Visaulization of Missing values BEFORE treatment:
--------------------------------------------------


In [8]:
# create a clean data log container for the cleaning process
cleaning_log = []
merged_data = merged_data.copy()


# Handle review date, check_out_date and check_in_date
merged_data['review_date'] = pd.to_datetime(merged_data['review_date'], errors="coerce").dt.normalize()
merged_data['check_in_date'] = pd.to_datetime(merged_data['check_in_date'], errors="coerce").dt.normalize()
merged_data['check_out_date'] = pd.to_datetime(merged_data['check_out_date'], errors="coerce").dt.normalize()
cleaning_log.append("Parsed data date to datetime; all values valid")


####---------Drop Duplicated and loyalty _status Columns:
"""Reason for dropping loyalty status is because its content is majorly missing data"""
merged_data = merged_data.drop(columns=["guest_id_booking_data", "hotel_id_booking_data","loyalty_status"])


###--------Cleaned Dateset Summary--------###
for line in cleaning_log:
    print("=>", line)

merged_data.head()

=> Parsed data date to datetime; all values valid


,review_id,booking_id,guest_id,hotel_id,review_text,rating_score,review_date,sentiment_label,room_type,booking_channel,...,check_out_date,total_amount,stay_duration,customer_type,country,previous_stays,hotel_name,location,star_rating,total_rooms
0,1,1,6699,18,The overall price was fair for what we paid. M...,3.2,2025-06-02,neutral,Suite,Direct Website,...,2025-05-26,296.44,1,Leisure,France,4,The Aberdeen Grand,Aberdeen,4,118
1,2,2,11380,23,The room was nothing out of the ordinary for w...,4.0,2025-01-22,positive,Deluxe,Corporate Portal,...,2025-01-13,483.56,4,Business,Germany,4,Nottingham Park Hotel,Nottingham,4,95
2,3,3,4812,13,"Unfortunately, the pool was outdated — it clea...",1.9,2025-03-13,negative,Executive,Travel Agent,...,2025-03-06,415.59,2,Business,Ireland,4,The London Boutique,London,4,231
3,4,4,982,12,We were disappointed that the neighborhood was...,2.2,2026-03-18,negative,Family Room,Airbnb,...,2026-03-09,601.18,3,Leisure,Nigeria,0,Manchester Metropolitan,Manchester,3,101
4,5,5,615,5,The room service was standard hotel fare for w...,3.3,2025-03-03,neutral,Deluxe,Expedia,...,2025-03-02,603.85,5,Group,United States,9,Glasgow Metropolitan,Glasgow,4,214


### Text Preprocessing

In [12]:
## Inspect the quality & characteristics of the review text
print(f"Missing Reviews Texts: {merged_data['review_text'].isna()}")
print(f"Duplicate Reviews: {merged_data['review_text'].duplicated().sum()}")

Missing Reviews Texts: 0        False
1        False
2        False
3        False
4        False
         ...  
34995    False
34996    False
34997    False
34998    False
34999    False
Name: review_text, Length: 35000, dtype: bool
Duplicate Reviews: 464


In [13]:
### Remove all duplicate reviews
review_corpus = merged_data.drop_duplicates(
    subset = ['review_text']).reset_index(drop=True)


### Remove all Whitespaces
review_corpus = review_corpus[review_corpus['review_text'].str.strip().ne("")].copy()

### check for len of reviews
review_corpus['text_length'] = review_corpus['review_text'].str.len()
review_corpus['text_length']



0         80
1        160
2        102
3        129
4        196
        ... 
34531    193
34532    144
34533    200
34534    136
34535    150
Name: text_length, Length: 34536, dtype: int64

In [14]:
lemmatizer = WordNetLemmatizer()
KEEP = {"no", "not", "nor", "never", "n't", "but", "very", "too"}     # do NOT remove negations!
STOPWORDS = set(ENGLISH_STOP_WORDS) - KEEP

###------- Create a function that cleans the review text
def light_clean_review(text):
    # convert text to string
    text = str(text)
    # remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)
    # remove all URL links
    text = re.sub(r'https?://\s+|www\.\s+', ' ', text)
    # # convert emojis to words
    # text = emoji.demojize(text, delimiters=(" ", " "))         
 
    # remove emojis
    text = text.encode("ascii", "ignore").decode('ascii')
    return text


def heavy_clean_review(text):
    # convert to lowercase
    text = light_clean_review(text).lower()
    # keep letter, number and space but remove special characters
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    # remove the remaining extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    # search of words  in stop_words
    words = [lemmatizer.lemmatize(w) for w in text.split() if w not in STOPWORDS and len(w) > 2]
    return " ".join(words)
    
merged_data["light_text"] = merged_data["review_text"].apply(light_clean_review)
merged_data["heavy_text"] = merged_data["review_text"].apply(heavy_clean_review)

merged_data[["review_text", "light_text", "heavy_text"]].head(10)

,review_text,light_text,heavy_text
0,The overall price was fair for what we paid. M...,The overall price was fair for what we paid. M...,overall price fair paid consider not
1,The room was nothing out of the ordinary for w...,The room was nothing out of the ordinary for w...,room ordinary paid overall price exceeded expe...
2,"Unfortunately, the pool was outdated — it clea...","Unfortunately, the pool was outdated it clear...",unfortunately pool outdated clearly needed upk...
3,We were disappointed that the neighborhood was...,We were disappointed that the neighborhood was...,disappointed neighborhood disappointingly way ...
4,The room service was standard hotel fare for w...,The room service was standard hotel fare for w...,room service standard hotel fare paid said bed...
5,Getting around from the hotel was in a rather ...,Getting around from the hotel was in a rather ...,getting hotel unsafe area really affected stay...
6,We couldn't have asked for a better base while...,We couldn't have asked for a better base while...,couldn asked better base visiting london plus ...
7,We had a wonderful time during our stay in Not...,We had a wonderful time during our stay in Not...,wonderful time stay nottingham plus surroundin...
8,The in-house cafe was acceptable for what we p...,The in-house cafe was acceptable for what we p...,house cafe acceptable paid said gym basic but ...
9,"The manager was fine, nothing special for what...","The manager was fine, nothing special for what...",manager fine special paid fitness facility spe...


In [15]:
# remove empty/very short reviews and keep English only
merged_data["n_words"] = merged_data["light_text"].str.split().str.len()
df = merged_data[merged_data["n_words"] >= 3]

def is_english(text):
    try: return detect(text) == "en"
    except Exception: return False

merged_data["is_english"] = merged_data["light_text"].apply(is_english)
print("English share:", merged_data["is_english"].mean().round(3))
merged_data = merged_data[merged_data["is_english"]].drop(columns="is_english")
print("After text cleaning:", len(merged_data))

English share: 1.0
After text cleaning: 35000


#### Sentiment Labels
Mapping rule (version this rule with your code!): **1-2 = negative, 3 = neutral, 4-5 = positive**.

> Spot-check 200-300

In [16]:
def rating_to_label(rating):
    return "negative" if rating <= 2 else "neutral" if rating == 3 else "positive"

merged_data["label"] = merged_data["rating_score"].apply(rating_to_label)
print(merged_data["label"].value_counts(normalize=True).round(3))

# manual spot-check sample -> open the CSV, read, and mark correct/incorrect
merged_data.sample(min(250, len(df)), random_state=42)[["review_id", "rating_score", "label", "review_text"]] \
  .to_csv("data/processed/label_spot_check.csv", index=False)

label
positive    0.803
negative    0.154
neutral     0.042
Name: proportion, dtype: float64


### Data Quality Validation

In [17]:
# Data  qaulity validation 
today = pd.Timestamp.today()
checks = {
    "review_id not null":        merged_data["review_id"].notna().all(),
    "review_id unique":          merged_data["review_id"].is_unique,
    "rating between 1 and 5":    merged_data["rating_score"].between(1, 5).all(),
    "text_light not empty":      (merged_data["light_text"].str.len() > 0).all(),
    "label in valid set":        merged_data["label"].isin(["negative", "neutral", "positive"]).all(),
    "review_date not in future": (merged_data["review_date"] <= today).all(),}

# print the result
for check, passed in checks.items():
    print("PASS" if passed else "FAIL", "-", check)

assert all(checks.values()), "Data quality checks failed - fix before modelling"

PASS - review_id not null
PASS - review_id unique
PASS - rating between 1 and 5
PASS - text_light not empty
PASS - label in valid set
PASS - review_date not in future


### Train / Validation / Test Split (70/15/15, stratified)
Guests can write several reviews, so we split by guest to avoid leakage, then check class balance.

In [18]:
# Split train data
train = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(train.split(merged_data, groups=merged_data["guest_id"]))
train, temp = merged_data.iloc[train_idx], merged_data.iloc[temp_idx]

# train data
test = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(test.split(temp, groups=temp["guest_id"]))
val, test = temp.iloc[val_idx], temp.iloc[test_idx]

for name, part in [("train", train), ("val", val), ("test", test)]:
    print(name, len(part), part["label"].value_counts(normalize=True).round(2).to_dict())

train 24557 {'positive': 0.81, 'negative': 0.15, 'neutral': 0.04}
val 5245 {'positive': 0.81, 'negative': 0.16, 'neutral': 0.04}
test 5198 {'positive': 0.79, 'negative': 0.16, 'neutral': 0.05}


In [19]:
merged_data.head()

,review_id,booking_id,guest_id,hotel_id,review_text,rating_score,review_date,sentiment_label,room_type,booking_channel,...,country,previous_stays,hotel_name,location,star_rating,total_rooms,light_text,heavy_text,n_words,label
0,1,1,6699,18,The overall price was fair for what we paid. M...,3.2,2025-06-02,neutral,Suite,Direct Website,...,France,4,The Aberdeen Grand,Aberdeen,4,118,The overall price was fair for what we paid. M...,overall price fair paid consider not,15,positive
1,2,2,11380,23,The room was nothing out of the ordinary for w...,4.0,2025-01-22,positive,Deluxe,Corporate Portal,...,Germany,4,Nottingham Park Hotel,Nottingham,4,95,The room was nothing out of the ordinary for w...,room ordinary paid overall price exceeded expe...,28,positive
2,3,3,4812,13,"Unfortunately, the pool was outdated — it clea...",1.9,2025-03-13,negative,Executive,Travel Agent,...,Ireland,4,The London Boutique,London,4,231,"Unfortunately, the pool was outdated it clear...",unfortunately pool outdated clearly needed upk...,15,negative
3,4,4,982,12,We were disappointed that the neighborhood was...,2.2,2026-03-18,negative,Family Room,Airbnb,...,Nigeria,0,Manchester Metropolitan,Manchester,3,101,We were disappointed that the neighborhood was...,disappointed neighborhood disappointingly way ...,21,positive
4,5,5,615,5,The room service was standard hotel fare for w...,3.3,2025-03-03,neutral,Deluxe,Expedia,...,United States,9,Glasgow Metropolitan,Glasgow,4,214,The room service was standard hotel fare for w...,room service standard hotel fare paid said bed...,39,positive


In [20]:
# save the Clean Dataset
# OUTPUT DIRECTORIES
OUT_DATA_DIR = Path("../Dataset/cleaned_data")
OUT_ARTIFACTS_DIR = Path("../Montra_Hospitality_Intel_PJ/artifacts")
OUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

# SAVE Data
merged_data.to_csv(OUT_DATA_DIR / "cleaned_montra_data.csv", index=False)
print("Merged Montra Data saved successfully!")
print("File:",(OUT_DATA_DIR / "cleaned_montra_data.csv").resolve())

Merged Montra Data saved successfully!
File: C:\Users\User\Documents\AMDARI_IMS\Montra_Hospitality_Intel_PJ\Dataset\cleaned_data\cleaned_montra_data.csv
